# Пояснительная записка  
## к дипломному проекту

# Система подбора аналогов вентиляционного оборудования на базе параметризатора и рекомендательных систем для интернет-магазина

**Учебный курс:** Data Scientist, расширенный курс - dsu-96  
**Учебная организация:** Нетология  
**Автор:** Чирков Виталий Вячеславович  
**Версия реализации:** **3.2.0**

---

## Содержание
1. Введение и постановка задачи
2. Описание данных и их особенностей
3. Описание обработки данных и разделение на обучающую и контрольную части
4. Описание возможных решений и итоговая архитектура
5. Описание параметризации, бизнес-логики, метрик и валидации
6. Итоговый результат, REST API и практическое применение
7. Заключение, выводы и планы развития
8. Список литературы
9. Приложения

---

## 1. Введение и постановка задачи

### 1.1 Контекст

Интернет-магазины климатического и вентиляционного оборудования работают с десятками тысяч наименований товаров. Значительная часть моделей является взаимозаменяемой: это могут быть товары разных производителей или разные серии одного бренда, предназначенные для похожего диапазона помещений и обладающие близкими техническими характеристиками.

Когда товар снимается с продажи, отсутствует на складе, меняется производитель или корректируется цена, покупателю и менеджеру требуется быстро подобрать замену. Такая замена должна быть не просто похожей по названию, но и допустимой по техническим параметрам.

Ручной подбор аналогов имеет несколько проблем:

- не масштабируется на большой каталог;
- зависит от опыта конкретного менеджера;
- может приводить к техническим ошибкам;
- медленно выполняется при большом количестве позиций;
- не позволяет систематически контролировать качество замен.

Поэтому возникает практическая задача автоматизировать подбор аналогов из каталога.

Для климатического оборудования текстового поиска недостаточно. Два товара могут иметь похожие названия, но различаться по мощности охлаждения, площади обслуживания, уровню шума, минимальной рабочей температуре или типу хладагента. Поэтому итоговая система должна опираться не только на текст, но и на нормализованные технические характеристики.

Дополнительной особенностью используемых данных является наличие поля `Analogs`, в котором для части товаров ранее были проставлены аналоги. Потенциально это поле могло бы использоваться как эталонное множество. Однако разведочный анализ показал, что поле содержит значительный шум и не может использоваться как единственный критерий качества в его исходном виде.

### 1.2 Бизнес-задача

Разработать программную систему, способную по строковому названию товара, артикулу или идентификатору возвращать отсортированный список релевантных аналогов из каталога с оценкой близости и объяснением совпадения.

Система должна:

1. находить товар по текстовому запросу, артикулу или идентификатору;
2. извлекать технические характеристики товара;
3. нормализовать характеристики к единым единицам измерения;
4. формировать кандидатов, потенциально подходящих в качестве аналогов;
5. оценивать каждого кандидата по согласованным бизнес-правилам;
6. возвращать первые `K` релевантных аналогов;
7. сохранять объяснения по каждому кандидату;
8. предоставлять REST API для интеграции с интернет-магазином.

### 1.3 Научная задача

Обосновать выбор комбинированного метода подбора аналогов, который объединяет:

- текстовое сходство нормализованных названий товаров;
- параметрическое сходство согласованных технических характеристик;
- бизнес-контроль допустимых замен;
- генерацию негативных пар для более надёжной оценки разделения классов «аналог / не аналог».

Целевая функция проекта формулируется как задача ранжирования релевантных кандидатов:

```text
FinalScore(query, candidate) = BusinessScore(query, candidate)
```

где:

- `query` - исходный товар;
- `candidate` - потенциальный аналог;
- `BusinessScore` - взвешенная оценка соответствия кандидата согласованным бизнес-правилам.

В ходе исследования было установлено, что поле `Analogs`:

- в полном объёме слишком шумное;
- но в своей технической части демонстрирует высокую согласованность с бизнес-правилами.

Это дало основание перейти от «чистых эвристик» к более сильной методологии:

## **План D**

Использовать `Analogs` как:

1. слабый источник позитивных пар;
2. диагностический источник шума;
3. базу для генерации негативных пар и проверки разделения классов.

## 1.4 Подходы к определению значимости характеристик

### План A - обучение на эталонной разметке

Если доступно достаточное количество чистых пар «исходный товар - аналог», можно построить модель классификации или ранжирования пар.

Пример:

```text
(query_product, candidate_product) → probability(analog)
```

Такой подход позволяет автоматически выявлять значимость признаков, но требует надёжной разметки.

### План B - бизнес-эвристики

Подход применяется, когда эталонная разметка отсутствует или является ненадёжной, но бизнес может согласовать критичные характеристики и допустимые отклонения.

Преимущества:

- объяснимость;
- быстрый запуск без обучения;
- возможность задать жёсткие ограничения;
- прозрачность для менеджеров и специалистов каталога.

Недостатки:

- веса и допуски могут быть субъективными;
- сложные нелинейные зависимости описываются вручную;
- требует экспертной проверки.

### План C - полуавтоматическое обучение на шумных метках

Получает высокую значимость в задачах со слабым supervised learning, но требует аккуратной очистки и валидации.

### План D - бизнес-эвристики + слабые позитивы + генерация негативных пар

Именно этот подход выбран итогово.

Суть:

1. бизнес-скоринг остаётся базовой объяснимой моделью;
2. поле `Analogs` используется не как истинный эталон в чистом виде, а как:
   - источник слабых позитивных пар;
   - источник диагностических наблюдений;
   - основа для формирования hard-negative кейсов.
3. отдельно генерируются негативные пары, чтобы:
   - проверить, насколько бизнес-score различает допустимые и недопустимые замены;
   - улучшить калибровку весов;
   - подготовить датасет для дальнейшей supervised / learning-to-rank модели.

### Почему План D лучше, чем просто B

Потому что теперь задача оценивается не только по вопросу:
- «сколько старых аналогов прошло бизнес-контроль?»

но и по более сильному вопросу:
- «насколько бизнес-score различает хорошие и плохие кандидаты?»

Это принципиально другой уровень методологической зрелости.

## 1.5 Ограничения

Использование методов глубокого обучения не требуется на этапе рабочей версии и сознательно исключено ради объяснимости, скорости и интерпретируемости для бизнеса.

Разработчик не имеет доступа к истории покупок и поведенческим данным пользователей, поэтому совместная фильтрация не применяется.

Работа ведётся на реальном датасете интернет-магазина в формате JSON. Данные содержат:

- массивы характеристик;
- неоднородные типы значений;
- смешанные форматы чисел;
- служебные пустые значения;
- неинформативные атрибуты;
- частично заполненные технические параметры;
- потенциально шумное поле аналогов.

Поле `Analogs` не является абсолютно достоверным, но в технической части показывает высокую согласованность с бизнес-логикой, что позволяет использовать его как слабый сигнал и как основу для негативной генерации.

Итоговый набор характеристик для подбора аналогов согласован с бизнесом:

- Инверторная технология
- Эффективен для помещ. площадью до
- Номинальная производительность охлаждения
- Номинальная производительность обогрева
- Уровень шума внутр. блока
- Мин. рабочая температура воздуха для внешнего блока
- Цвет корпуса внутр. блока
- Wi-Fi модуль
- Управление с мобильного приложения по Wi-Fi
- Хладагент
- Класс энергоэффективности

## 1.6 Метрики качества решения

### 1.6.1 Доля валидных значений

Для признака `f`:

```text
ValidCoverage(f) =
    число записей с корректным значением f
    /
    общее число записей в выборке
```

### 1.6.2 Бизнес-оценка пары товаров

```text
BusinessScore(q, c) =
    Σ weight_i * ComponentScore_i(q, c)
    /
    Σ weight_i
```

где:

- `ComponentScore_i ∈ [0, 1]`.

### 1.6.3 Разделение пар на проверяемые и непроверяемые

Для каждой пары вычисляется:

- `is_evaluable`;
- число валидных признаков;
- число валидных core-признаков;
- confidence пары.

Пара считается **проверяемой**, если:
- есть минимум два валидных согласованных признака;
- есть минимум два валидных core-технических признака из:
  - `cooling_power_kw`
  - `area_m2`
  - `heating_power_kw`

### 1.6.4 Доля проверяемых пар

```text
EvaluableRate =
    число проверяемых пар / общее число пар
```

### 1.6.5 Бизнес-проходимость по всем и по проверяемым парам

```text
PassRateAll =
    число прошедших пар / все пары

PassRateEvaluable =
    число прошедших пар / только проверяемые пары
```

Это критически важная методологическая поправка:
- `PassRateAll` показывает общий шум;
- `PassRateEvaluable` показывает, насколько поле согласуется с бизнес-логикой там, где вообще можно технически сравнить товары.

### 1.6.6 Покрытие кандидатов истинными аналогами

```text
CandidateRecall =
    число запросов, где хотя бы один истинный аналог попал в candidate set
    /
    число запросов с непустым TrueAnalogs
```

### 1.6.7 Бизнес-качество candidate set

```text
BusinessPassRateOnCandidates =
    число кандидатов с score >= threshold и без hard_fail
    /
    общее число оценённых кандидатов
```

```text
CriticalRejectRateOnCandidates =
    число кандидатов с hard_fail
    /
    общее число проверенных кандидатов
```

### 1.6.8 Метрики разделения позитивов и негативов для Плана D

Для набора:

- `PositivePairs`:
  - слабые позитивные пары из `Analogs`, прошедшие бизнес-контроль;
- `NegativePairs`:
  - rule-based, retrieval-based и synthetic негативные пары.

Используются метрики:

- `AUROC`
- `AUPRC`
- `PairwiseAccuracy`
- `MeanPositiveScore`
- `MeanNegativeScore`
- `FalsePositiveRate`
- `NegativeCaptureRate`

## 1.7 Фактические результаты

### 1.7.1 Объём данных

| Показатель | Значение |
| :--- | ---: |
| Загружено записей | 8 591 |
| После фильтрации по категории | 7 339 |
| Товаров с `Analogs` в core-выборке | 1 168 |
| Истинных аналоговых пар | 16 407 |
| Уникальных целей в `Analogs` | 439 |
| Уникальных целей, присутствующих в core | 435 |

Это важная деталь:
- поле `Analogs` содержит большое число перекрёстных ссылок;
- уникальных целевых товаров в аналогах существенно меньше, чем самих пар;
- это подтверждает, что поле заполнено не по принципу «полноценной матрицы замен», а частично / агрегированно.

### 1.7.2 Доля валидных значений согласованных характеристик

| Характеристика | Поле | Важность | Доля валидных значений |
| :--- | :--- | ---: | ---: |
| Номинальная производительность охлаждения | `cooling_power_kw` | 3 | 0.5850 |
| Уровень шума внутр. блока | `noise_indoor_db` | 3 | 0.5850 |
| Мин. рабочая температура воздуха для внешнего блока | `min_operating_temp_c` | 3 | 0.5843 |
| Номинальная производительность обогрева | `heating_power_kw` | 3 | 0.5839 |
| Эффективен для помещ. площадью до | `area_m2` | 3 | 0.5799 |
| Хладагент | `refrigerant` | 2 | 0.5396 |
| Цвет корпуса внутр. блока | `color_inner_body` | 3 | 0.3536 |
| Класс энергоэффективности | `energy_class` | 2 | 0.3465 |
| Управление с мобильного приложения по Wi-Fi | `has_wifi_app_control` | 1 | 0.2763 |
| Wi-Fi модуль | `has_wifi_module` | 1 | 0.0977 |
| Инверторная технология | `is_inverter` | 3 | 0.3409 |

Основные числовые характеристики заполнены примерно на 58%. Этого достаточно для построения объяснимой бизнес-оценки, но недостаточно для полностью автоматической эксплуатации без экспертной проверки.

### 1.7.3 Разведочный анализ поля `Analogs`

| Показатель | Значение |
| :--- | ---: |
| Всего пар | 16 407 |
| Проверяемые пары | 8 245 |
| Непроверяемые пары | 8 162 |
| `EvaluableRate` | 0.5025 |
| Прошли бизнес-контроль среди всех пар | 8 175 |
| `PassRateAll` | 0.4983 |
| Прошли бизнес-контроль среди проверяемых пар | 8 175 |
| `PassRateEvaluable` | 0.9915 |
| Средний score по проверяемым парам | 0.7979 |
| Средний confidence по проверяемым парам | 0.8034 |

### Вывод
Это ключевой момент всей работы:

> Поле `Analogs` нельзя использовать как чистый эталон,  
> но в технической части оно почти полностью проходит согласованный бизнес-контроль.

То есть проблема не столько в том, что аналоговые метки «вымышленные»,  
сколько в том, что значительная часть пар не поддалась технической оценке из-за пропусков.

### 1.7.4 Итоговый режим работы

| Показатель | Значение |
| :--- | :--- |
| `USE_TRUE_ANALOGS_VALIDATION` | false |
| `USE_WEAK_ANALOGS_SOURCE` | true |
| `selected_mode` | `B_business_heuristics_with_weak_evaluable_source` |
| пороги | `MIN_EVALUABLE_RATE=0.60`, `WEAK_EVALUABLE_RATE=0.45`, `MIN_PASS_RATE_EVALUABLE=0.55`, `WEAK_PASS_RATE_EVALUABLE=0.90` |

### Почему так
Потому что:

- полная надёжность `Analogs` не подтверждена;
- но качество внутри проверяемой части очень высокое;
- поэтому поле можно использовать как:
  - слабый источник позитивов;
  - источник генерации негативов;
  - источник валидации разделения.

### 1.7.5 Retrieval-метрики по `Analogs`

| Метрика | Значение |
| :--- | ---: |
| Количество запросов с `Analogs` | 1 168 |
| MRR | 0.2692 |
| Precision@1 | 0.1421 |
| Recall@1 | 0.0290 |
| HitRate@1 | 0.1421 |
| Precision@3 | 0.1347 |
| Recall@3 | 0.0735 |
| HitRate@3 | 0.3022 |
| Precision@5 | 0.1519 |
| Recall@5 | 0.1267 |
| HitRate@5 | 0.4486 |
| Precision@10 | 0.1903 |
| Recall@10 | 0.2842 |
| HitRate@10 | 0.6678 |

Это уже принципиально другой уровень по сравнению с предыдущей версией работы.

### Вывод
Ранее retrieval-метрики были практически нулевыми.  
После переработки:

- unit-aware нормализации;
- пересмотра business-score;
- изменения candidate generation;
- корректной обработки пропусков;

качество retrieval выросло кратно.

Но важно:
- `Analogs` всё ещё шумный;
- retrieval-метрики остаются диагностическими;
- они не являются единственным критерием промышленной годности системы.

### 1.7.6 Candidate generation

| Метрика | Значение |
| :--- | ---: |
| `max_candidates` | 300 |
| `total_queries` | 1 168 |
| `queries_with_candidate_hit` | 1 151 |
| `candidate_recall` | 0.9854 |
| `total_true_pairs` | 16 407 |
| `covered_true_pairs` | 7 435 |
| `true_analog_pair_coverage` | 0.4532 |
| `average_candidate_count` | 300 |

### Что это означает
Кандидатный слой работает отлично на уровне запросов:

- почти для 99% запросов хотя бы один истинный аналог попадает в candidate set.

Но на уровне всех истинных пар покрытие умеренное:
- около 45%.

Причины:
- большое число пар;
- высокая дублированность целей;
- часть истинных аналогов не попадает в близкий размерный класс / текстовую близость.

### 1.7.7 Бизнес-качество candidate set

| Метрика | Значение |
| :--- | ---: |
| threshold | 0.60 |
| max_candidates | 300 |
| evaluated_queries | 300 |
| total_candidates_evaluated | 90 000 |
| passed_candidates | 23 995 |
| critical_reject_candidates | 48 862 |
| business_pass_rate_on_candidates | 0.2666 |
| critical_reject_rate_on_candidates | 0.5429 |
| average_candidate_score | 0.6125 |
| top1_pass_rate | 0.8933 |

### Вывод
Здесь видна реальная производственная картина:

1. candidate set большой;
2. много кандидатов технически проходят бизнес-допуск;
3. значительная доля кандидатов отбраковывается как критически несоответствующие;
4. top-1 у 89% запросов проходит бизнес-порог.

Это означает, что система уже работает как полезный ассистент подбора замен,  
но её качество нужно дополнительно оценивать не только retrieval-метриками, а именно разделением «хороший / плохой кандидат».

## 1.8 План D: генерация негативных пар

### 1.8.1 Идея
Если нет полноценной размеченной выборки «аналог / не аналог», можно:

1. взять слабые позитивы из `Analogs`;
2. сгенерировать негативы тремя способами:
   - rule-based синтетические;
   - retrieval hard negatives;
   - cross-pair negatives;
3. проверить, насколько бизнес-score разделяет эти два класса.

### 1.8.2 Позитивные пары

Позитивная пара — это пара `(query, candidate)`, для которой:

- `candidate ∈ Analogs(query)`;
- пара технически проверяема;
- `BusinessScore >= 0.60`;
- `hard_fail = False`;
- есть минимум 2 core-технических поля.

Формально:

```text
PositivePair(q, c) =
    c ∈ Analogs(q)
    AND is_evaluable(q, c) = True
    AND BusinessScore(q, c) >= 0.60
    AND hard_fail(q, c) = False
    AND core_valid_field_count(q, c) >= 2
```

### 1.8.3 Негативные пары

Негативные пары генерируются тремя источниками.

### Источник 1. Rule-based синтетические негативы
Для реальной позитивной пары `(q, c)` искусственно нарушается один или два критичных признака:

- мощность охлаждения уходит за допуск;
- мощность обогрева уходит за допуск;
- площадь уходит за допуск;
- инверторность инвертируется;
- цвет меняется на несовместимый;
- хладагент меняется;
- температура становится недопустимо высокой.

### Источник 2. Retrieval hard negatives
Для каждого запроса берутся кандидаты из retrieval-слоя:

- не входящие в `Analogs(query)`;
- технически проверяемые;
- имеющие низкий бизнес-score или `hard_fail`.

Наиболее безопасный критерий:

```text
NegativeHard(q, c) =
    c ∈ CandidateSet(q)
    AND c ∉ Analogs(q)
    AND is_evaluable(q, c) = True
    AND (
        BusinessScore(q, c) < 0.50
        OR hard_fail(q, c) = True
    )
```

### Источник 3. Cross-pair negatives
Случайные проверяемые пары из одного каталога, которые:
- не являются позитивами;
- имеют низкий score;
- нарушают критические признаки.

### 1.8.4 Зачем нужны негативы
Они позволяют ответить на более сильный вопрос:

> Если бизнес-score действительно хорошо измеряет качество замены,  
> то он должен уверенно различать позитивы и негативы.

Для этого оценивается разделение классов:
- распределение score у позитивов;
- распределение score у негативов;
- метрики AUROC / AUPRC / accuracy.

### 1.8.5 Итоговая роль Плана D
План D не заменяет бизнес-скоринг.

Он делает систему:
- более оцениваемой;
- более калибруемой;
- готовой к переходу от эвристик к supervised / learning-to-rank.

## 1.9 Ссылки на репозиторий и ноутбук разведочного анализа

Место для ссылки на репозиторий с кодом приложения:

```text
https://github.com/sladeshka/thesis.netology
```

Место для ссылки на ноутбук разведочного анализа:

```text
https://github.com/sladeshka/thesis.netology/blob/main/notebooks/eda_analog_selection.ipynb
```

Место для ссылки на артефакты запуска:

```text
https://github.com/sladeshka/thesis.netology/tree/main/artifacts
```

---

# 2. Описание данных и их особенностей

## 2.1 Источник и объём

Используется реальный датасет номенклатуры интернет-магазина климатического оборудования в формате JSON.

| Показатель | Значение |
| :--- | ---: |
| Загружено записей | 8 591 |
| Core-выборка после фильтрации по категории | 7 339 |

## 2.2 Структура одной записи

Каждая запись содержит уникальный идентификатор `Id`, название товара `Name`, артикул `Code`, массив характеристик `Characteristics` и опционально список аналогов `Analogs`.

Пример структуры:

```json
{
  "Id": "01M1Y01TZT0Z3EA653657WBG6C",
  "Name": "Сплит-система инверторного типа ...",
  "Code": "НС-1771518",
  "Characteristics": [
    {
      "Name": "Эффективен для помещ. площадью до",
      "Value": "22"
    },
    {
      "Name": "Номинальная производительность охлаждения",
      "Value": "2.2"
    }
      ...
  ],
  "Analogs": [
    {
      "ProductId": "01KTSCSP4DF7R1YRY9DJZ4R678",
      "MatchScore": 100,
      "Source": 1
    }
  ]
}
```

## 2.3 Особенности массива характеристик

Массив характеристик содержит большое число потенциальных ключей. Значительная часть из них не используется для задачи подбора аналогов:

- ссылки на фотографии;
- ссылки на видео;
- инструкции;
- сертификаты;
- декларации;
- BIM-модели;
- 3D-модели;
- HTML-описания;
- маркетинговые атрибуты;
- служебные поля.

Это подтверждает необходимость фильтрации и ограниченного словаря признаков.

## 2.4 Единицы измерения числовых признаков

Используемые числовые признаки:

- `area_m2`
- `cooling_power_kw`
- `heating_power_kw`
- `noise_indoor_db`
- `min_operating_temp_c`

Итоговые единицы измерения:

| Характеристика | Единица измерения |
| :--- | :--- |
| Эффективен для помещ. площадью до | м² |
| Номинальная производительность охлаждения | кВт |
| Номинальная производительность обогрева | кВт |
| Уровень шума внутр. блока | дБ(А) |
| Мин. рабочая температура воздуха для внешнего блока | °C |

Правило нормализации мощности:

```text
cooling_power_kw:
    значение <= 100 → кВт
    значение > 100 → Вт → кВт

heating_power_kw:
    значение <= 100 → кВт
    значение > 100 → Вт → кВт
```

Формула:

```python
kW = W / 1000
```

## 2.5 Проблема ложной заполненности категориальных признаков

В первичном разведочном анализе многие категориальные признаки имели формальную заполненность около 100%, одновременно с доминирующим значением `nan`.

Поэтому используется метрика:

```text
доля валидных значений
```

## 2.6 Качество поля `Analogs`

Поле `Analogs` потенциально могло служить эталонным множеством.

Результаты диагностики:

| Показатель | Значение |
| :--- | ---: |
| Товаров с `Analogs` в raw-выборке | 1 583 |
| Товаров с валидными аналогами в core-выборке | 1 168 |
| Пар в core-выборке | 16 407 |
| Уникальных целей в `Analogs` | 439 |
| Уникальных целей, присутствующих в core | 435 |

Это означает, что:
- поле заполнено не полностью;
- ссылки часто дублируются;
- оно не является полноценной матрицей взаимозаменяемости.

## 2.7 Итог раздела

Данные пригодны для построения системы подбора аналогов после многоступенчатой предобработки.

Основная особенность:
- техническая часть данных достаточно чистая, чтобы считать бизнес-score;
- разметка `Analogs` достаточно хорошая в технической части, чтобы использовать её как слабый источник позитивов и генерации негативов.

---

# 3. Описание обработки данных и разделение на обучающую и контрольную части

## 3.1 Конфигурация

Все параметры обработки зафиксированы в конфигурации модуля. Ключевые параметры:

- случайное состояние;
- доли контрольной и диагностической выборок;
- список согласованных бизнес-характеристик;
- веса важности характеристик;
- допустимые отклонения;
- параметры TF-IDF;
- количество кандидатов на запрос;
- пороги качества;
- пути артефактов.

## 3.2 Загрузка

JSON-файл читается через поток с детекцией формата: массив записей или потоковый NDJSON. Поддерживается лимит записей для разработки и ограничения объёма памяти.

## 3.3 Фильтрация характеристик

Из массива характеристик исключаются:

- ссылки на фото;
- ссылки на видео;
- инструкции;
- сертификаты;
- декларации;
- BIM-модели;
- 3D-модели;
- HTML-описания;
- маркетинговые атрибуты;
- служебные поля;
- пустые и неинформативные значения.

## 3.4 Нормализация числовых значений

Числовые признаки проходят через парсер с учётом единиц измерения.

Обработаны:

- запятая как десятичный разделитель;
- неразрывные пробелы;
- пробелы как разделители тысяч;
- Unicode-минус;
- смешанные форматы записи;
- строки с текстовыми пояснениями;
- кВт и Вт для мощности.

## 3.5 Нормализация категориальных и булевых признаков

Булевые признаки нормализуются к `True`, `False` или `None`.

Поддерживаются варианты:

- Да / Нет
- true / false
- 1 / 0
- Есть / Нет
- В наличии / Нет

Категориальные признаки нормализуются:

- приводятся к единому регистру;
- удаляются лишние пробелы;
- схлопываются синонимы;
- неинформативные значения переводятся в пустые.

Для цвета корпуса выполняется приведение синонимов.  
Для класса энергоэффективности используется порядковая нормализация.

## 3.6 Фильтрация по категории

Фильтр применяется к полю `Name` с регистронезависимым поиском ключевых слов:

- сплит
- кондицион
- кондиционер

## 3.7 Разделение на обучающую и контрольную части

Для retrieval-задачи обычное разбиение на обучающую и тестовую выборки применяется не напрямую к товарам, а к запросам.

Корректно разделять идентификаторы товаров-запросов:

```text
diagnostic_query_ids
control_query_ids
```

Текущее разбиение:

| Показатель | Значение |
| :--- | ---: |
| Всего запросов с `Analogs` | 1 168 |
| diagnostic queries | 934 |
| control queries | 234 |
| random_state | 42 |

Разделение используется как диагностическое:
- для проверки retrieval-слоя;
- для диагностики бизнес-score;
- для оценки разделения классов в Плане D.

## 3.8 Конвейер признаков

Конвейер выполняет:

1. загрузку JSON;
2. валидацию записей;
3. извлечение характеристик;
4. нормализацию единиц измерения;
5. нормализацию категорий;
6. построение профиля товара;
7. формирование candidate set;
8. расчёт бизнес-оценки;
9. ранжирование первых `K` аналогов;
10. сохранение артефактов;
11. генерацию негативных пар для Плана D.

# 4. Описание возможных решений и итоговая архитектура

## 4.1 Альтернативные подходы

### 4.1.1 Текстовый поиск
- плюсы: простота, скорость;
- минусы: не гарантирует техническую совместимость.

### 4.1.2 Параметрический подход
- плюсы: высокая применимость, объяснимость, бизнес-контроль;
- минусы: требует качественной нормализации.

### 4.1.3 Нейросетевой эмбеддинговый подход
- плюсы: потенциально лучше понимает семантику;
- минусы: сложнее интерпретировать, дороже inference, требует разметки.

### 4.1.4 Совместная фильтрация
- плюсы: поведенческая релевантность;
- минусы: нет истории покупок, риск технических ошибок.

## 4.2 Сравнение подходов

| Подход | Точность в задаче | Объяснимость | Стоимость | Требуемые данные |
| :--- | :--- | :--- | :--- | :--- |
| Текстовый TF-IDF | средняя | высокая | низкая | названия |
| Параметрический бизнес-score | высокая на структурированных данных | высокая | средняя | характеристики |
| Нейросетевой | потенциально высокая | низкая | высокая | тексты, разметка |
| Совместный | потенциально высокая | низкая | средняя | история покупок |

## 4.3 Выбранный подход

Выбран гибридный retrieval-подход с бизнес-score и Планом D:

```text
CandidateSet(q) = SizeBucketCandidates(q) ∪ TextSimilarityCandidates(q)

FinalScore(q, c) = BusinessScore(q, c), где c ∈ CandidateSet(q)

Filter(q, c) =
    exclude if hard_fail(q, c)
    exclude if BusinessScore(q, c) < min_score
```

Текстовая компонента применяется как вспомогательная:
- поиск товара;
- расширение кандидатов;
- первичное сокращение перебора.

Бизнес-скоринг — итоговая основа ранжирования.

План D — слой оценки и калибровки:
- генерация негативных пар;
- проверка разделения классов;
- подготовка к будущему supervised learning.

## 4.4 Согласованные бизнес-характеристики

Используется следующий набор:

```json
[
  {
    "Name": "Инверторная технология",
    "Field": "is_inverter",
    "Kind": "bool",
    "Importance": 3,
    "TolerancePercent": null
  },
  {
    "Name": "Эффективен для помещ. площадью до",
    "Field": "area_m2",
    "Kind": "numeric_relative",
    "Importance": 3,
    "TolerancePercent": 15
  },
  {
    "Name": "Номинальная производительность охлаждения",
    "Field": "cooling_power_kw",
    "Kind": "numeric_relative",
    "Importance": 3,
    "TolerancePercent": 15
  },
  {
    "Name": "Номинальная производительность обогрева",
    "Field": "heating_power_kw",
    "Kind": "numeric_relative",
    "Importance": 3,
    "TolerancePercent": 15
  },
  {
    "Name": "Уровень шума внутр. блока",
    "Field": "noise_indoor_db",
    "Kind": "numeric_relative",
    "Importance": 3,
    "TolerancePercent": 40
  },
  {
    "Name": "Мин. рабочая температура воздуха для внешнего блока",
    "Field": "min_operating_temp_c",
    "Kind": "temp_relative",
    "Importance": 3,
    "TolerancePercent": 30
  },
  {
    "Name": "Цвет корпуса внутр. блока",
    "Field": "color_inner_body",
    "Kind": "category_exact",
    "Importance": 3,
    "TolerancePercent": null
  },
  {
    "Name": "Wi-Fi модуль",
    "Field": "has_wifi_module",
    "Kind": "bool",
    "Importance": 1,
    "TolerancePercent": null
  },
  {
    "Name": "Управление с мобильного приложения по Wi-Fi",
    "Field": "has_wifi_app_control",
    "Kind": "bool",
    "Importance": 1,
    "TolerancePercent": null
  },
  {
    "Name": "Хладагент",
    "Field": "refrigerant",
    "Kind": "category_exact",
    "Importance": 2,
    "TolerancePercent": null
  },
  {
    "Name": "Класс энергоэффективности",
    "Field": "energy_class",
    "Kind": "energy_class",
    "Importance": 2,
    "TolerancePercent": null
  }
]
```

Важность трактуется согласно классификации:

```text
NoMatter = 0
Matter = 1
Important = 2
Critical = 3
```

Используемые веса:

```text
Critical = 1.00
Important = 0.75
Matter = 0.25
NoMatter = 0.00
```

После нормировки итоговые веса:

| Поле | Вес |
| :--- | ---: |
| `is_inverter` | 0.111111 |
| `area_m2` | 0.111111 |
| `cooling_power_kw` | 0.111111 |
| `heating_power_kw` | 0.111111 |
| `noise_indoor_db` | 0.111111 |
| `min_operating_temp_c` | 0.111111 |
| `color_inner_body` | 0.111111 |
| `energy_class` | 0.083333 |
| `refrigerant` | 0.083333 |
| `has_wifi_app_control` | 0.027778 |
| `has_wifi_module` | 0.027778 |

## 4.5 Формула бизнес-оценки

```text
score =
    Σ(weight_i * component_score_i)
    /
    Σ(weight_i)
```

где:

```text
component_score_i ∈ [0, 1]
```

Компонентная оценка зависит от типа характеристики:
- для числовых с процентным допуском оценивается отклонение;
- для булевых - совпадает значение или нет;
- для категориальных - точное совпадение;
- для класса энергоэффективности - учитывается порядковая шкала.

## 4.6 Обработка процентных допусков

Для числовых характеристик допустимая разница рассчитывается так:

```text
reference = max(abs(query_value), 1.0)

allowed_diff = reference * tolerance_percent / 100
```

Если абсолютная разница не превышает допустимую, компонент получает оценку 1.  
Если превышает, оценка снижается или может стать нулевой.

Для температуры превышение допуска может рассматриваться как критическое несоответствие.

## 4.7 Обработка пропусков

Если значение характеристики отсутствует у одного из товаров, компонент получает нейтральную оценку:

```text
component_score = 0.5
```

Это позволяет системе корректно работать на неполных карточках, но снижает уверенность итоговой рекомендации.

## 4.8 Итоговая архитектура

Система построена как конвейер из следующих компонентов.

### Препроцессор
- нормализация названий;
- очистка значений;
- приведение типов;
- обработка Unicode;
- фильтрация служебных значений.

### Параметризатор
- построение профилей товаров;
- нормализация единиц измерения;
- формирование матрицы признаков.

### Текстовый модуль
- TF-IDF по названиям;
- поиск товара по строковому запросу;
- текстовая близость кандидатов.

### Candidate generator
- size_bucket candidates;
- TF-IDF text candidates;
- ограниченное число кандидатов на запрос.

### Бизнес-скоринг
- расчёт близости пары товаров;
- возврат:
  - score
  - hard_fail
  - component_details
  - confidence

### Ранжирование
- сортировка по бизнес-оценке;
- отсев критических несоответствий;
- фильтрация по `min_score`.

### Plan D layer
- построение позитивных пар;
- генерация негативных пар;
- оценка разделения классов;
- сохранение артефактов.

### Сервисный слой
- FastAPI;
- REST API;
- объяснимый ответ.

## 4.9 Архитектурная схема

```text
Вход: ID / артикул / текстовый запрос
        ↓
Поиск товара в каталоге
        ↓
Извлечение и нормализация профиля товара
        ↓
Формирование кандидатов:
  - size_bucket
  - TF-IDF text similarity
        ↓
BusinessScore по согласованным характеристикам
        ↓
Отсев hard_fail
        ↓
Фильтр score >= min_score
        ↓
Ранжирование первых K аналогов
        ↓
Ответ:
  - id / name / code
  - score
  - text_similarity
  - component_details
  - reasons
  - ключевые характеристики

Параллельно:
  - positive pair mining
  - negative pair generation
  - separation evaluation
  - model calibration
```

## 4.10 Обоснование отказа от глубокого обучения

Отказ от нейросетей в рабочей версии аргументирован структурированностью данных, интерпретируемостью, скоростью итераций, низким порогом входа для поддержки и соответствием бизнес-требованиям.

Архитектура допускает добавление текстовых эмбеддингов как дополнительной компоненты без переписывания остальных модулей.

## 4.11 Оценка сложности

На текущей подвыборке скоринг выполняется за время, линейное от количества кандидатов. Для масштаба 100 тысяч активных позиций рекомендуется:

- использовать индекс candidates по `size_bucket`;
- использовать sparse TF-IDF retrieval;
- перейти к векторному индексу для текстовой компоненты;
- кэшировать результаты по артикулу;
- ограничивать глубину перебора кандидатов;
- использовать асинхронный API и фоновую инициализацию каталога.

---

# 5. Описание параметризации, бизнес-логики, метрик и валидации

## 5.1 Выбор режима работы

Перед запуском бизнес-скоринга выполнялась проверка качества поля `Analogs`.

Итоговые показатели:

| Показатель | Значение |
| :--- | ---: |
| Всего пар | 16 407 |
| Проверяемые пары | 8 245 |
| `EvaluableRate` | 0.5025 |
| `PassRateAll` | 0.4983 |
| `PassRateEvaluable` | 0.9915 |

Итоговый режим:

```text
selected_mode = B_business_heuristics_with_weak_evaluable_source
```

Причина:

```text
Полный Analogs нельзя использовать как чистый эталон из-за низкой проверяемости,
но среди технических пар его качество высокое.
Можно использовать проверяемую часть как слабую разметку
и дополнительно строить негативные пары.
```

## 5.2 Итоговые бизнес-параметры

Итоговые веса:

```text
is_inverter: 0.111111
area_m2: 0.111111
cooling_power_kw: 0.111111
heating_power_kw: 0.111111
noise_indoor_db: 0.111111
min_operating_temp_c: 0.111111
color_inner_body: 0.111111
has_wifi_module: 0.027778
has_wifi_app_control: 0.027778
refrigerant: 0.083333
energy_class: 0.083333
```

Итоговые допуски:

| Характеристика | Допуск |
| :--- | ---: |
| Эффективен для помещ. площадью до | 15% |
| Номинальная производительность охлаждения | 15% |
| Номинальная производительность обогрева | 15% |
| Уровень шума внутр. блока | 40% |
| Мин. рабочая температура воздуха для внешнего блока | 30% |

## 5.3 Кандидатная генерация

Для каждого товара формируются кандидаты по двум источникам:

1. товары из того же размерного класса;
2. товары с наибольшей текстовой близостью.

Если истинный аналог не попал в candidate set, retrieval-метрики будут низкими независимо от качества бизнес-score. Поэтому отдельно отслеживается:

```text
CandidateRecall
```

Результат:
- `candidate_recall = 0.9854`

Это означает, что retrieval-слой в целом хорошо покрывает запросы с истинными аналогами.

## 5.4 Валидация через `Analogs`

Итоговые retrieval-метрики:

| Метрика | Значение |
| :--- | ---: |
| n_queries | 1 168 |
| MRR | 0.2692 |
| Precision@1 | 0.1421 |
| Recall@1 | 0.0290 |
| HitRate@1 | 0.1421 |
| Precision@3 | 0.1347 |
| Recall@3 | 0.0735 |
| HitRate@3 | 0.3022 |
| Precision@5 | 0.1519 |
| Recall@5 | 0.1267 |
| HitRate@5 | 0.4486 |
| Precision@10 | 0.1903 |
| Recall@10 | 0.2842 |
| HitRate@10 | 0.6678 |

Несмотря на шум `Analogs`, качество заметно выросло после доработки конвейера.

## 5.5 Бизнес-валидация кандидатов

Итоговые метрики:

| Метрика | Значение |
| :--- | ---: |
| threshold | 0.60 |
| evaluated_queries | 300 |
| total_candidates_evaluated | 90 000 |
| passed_candidates | 23 995 |
| critical_reject_candidates | 48 862 |
| business_pass_rate_on_candidates | 0.2666 |
| critical_reject_rate_on_candidates | 0.5429 |
| average_candidate_score | 0.6125 |
| top1_pass_rate | 0.8933 |

Это означает:
- система реально оценивает кандидатов;
- top-1 достаточно часто проходит бизнес-порог;
- но общий candidate set содержит много технического шума;
- поэтому нужен План D.

## 5.6 Чувствительность к числовым изменениям

Для проверки robustness выполняется sensitivity analysis: для реальных пар «запрос - кандидат» тестируется изменение числовых признаков на:

- +5%
- +10%
- +20%
- +30%

Итог:
- умеренные отклонения слабо ломают score;
- значительные отклонения уже заметно снижают оценку;
- для температуры появляются первые hard_fail.

Это подтверждает, что система не является «хрупкой»,  
но при сильных нарушениях бизнес-допусков оценка падает.

## 5.7 План D: позитивные и негативные пары

### Позитивные пары
Формируются из:
- `Analogs`;
- проверяемых пар;
- пар с `BusinessScore >= 0.60`.

### Негативные пары
Формируются из:
- retrieval-кандидатов с низким score;
- rule-based синтетических нарушений;
- cross-pair случайных пар.

### Цель
Проверить, насколько бизнес-score разделяет два класса:
- хорошие аналоги;
- недопустимые замены.

Ожидаемые артефакты:
- `positive_pairs.csv`
- `negative_pairs.csv`
- `plan_d_separation_metrics.json`
- визуализация распределений score

## 5.8 Объяснимость

Для каждого кандидата сохраняется структура детализации признаков, где видно:

- вклад каждого признака;
- вес признака;
- компонентную оценку;
- признак жёсткого ограничения;
- человекочитаемые причины.

Пример логики объяснения:

```text
Низкая оценка из-за мощности охлаждения: отклонение больше 10%.
Совпадение по площади и температуре повышает итоговую оценку.
Отсутствие Wi-Fi модуля снижает оценку, но не является автоматическим отвержением.
```

---

# 6. Итоговый результат, REST API и практическое применение

## 6.1 Итоговый артефакт

Итогом работы является программный комплекс, включающий:

- модули предобработки;
- параметризатор;
- текстовый поиск;
- бизнес-скоринг;
- retrieval-оценку;
- FastAPI-сервис;
- набор тестов;
- ноутбуки разведочного анализа;
- артефакты запуска.

## 6.2 REST API

Сервис предоставляет следующие эндпоинты.

| Эндпоинт | Метод | Назначение |
| :--- | :--- | :--- |
| `/health` | GET | Проверка готовности сервиса |
| `/stats` | GET | Статистика инициализации, веса, покрытие |
| `/business-parameters` | GET | Возвращает веса, допуски и характеристики бизнес-модели |
| `/predict` | POST | Подбор аналогов по ID, артикулу или тексту |
| `/predict_by_id` | POST | Подбор аналогов по товарному ID |
| `/categories` | GET | Список категорий |

## 6.3 Контракт запроса

```json
{
  "query": "01KTTYDTX5H6JKKW2XX5Q3YDC7",
  "k": 5,
  "min_score": 0.6,
  "include_details": true
}
```

## 6.4 Контракт ответа

```json
{
  "query": "01KTTYDTX5H6JKKW2XX5Q3YDC7",
  "query_id": "01KTTYDTX5H6JKKW2XX5Q3YDC7",
  "query_type": "id",
  "k": 5,
  "results": [
    {
      "id": "01KTTYDTSM36WC6501V0RD7E8D",
      "code": null,
      "name": "Инверторная сплит система Mitsubishi SRK20ZS-W Premium (Белый)",
      "score": 0.8210,
      "text_similarity": 0.82,
      "is_inverter": true,
      "area_m2": 20.0,
      "cooling_power_kw": 2.0,
      "heating_power_kw": 2.7,
      "noise_indoor_db": 19.0,
      "min_operating_temp_c": -15.0,
      "color_inner_body": null,
      "has_wifi_module": null,
      "has_wifi_app_control": null,
      "refrigerant": "r32",
      "energy_class": null,
      "component_details": {},
      "reasons": []
    }
  ],
  "total_candidates_evaluated": 200,
  "catalog_size": 6054,
  "execution_time_ms": 12.34,
  "timestamp": "2024-01-01T00:00:00"
}
```

## 6.5 Практическое применение

### Сценарий 1. Блок похожие модели в карточке товара
Для покупателя показываются технически обоснованные аналоги текущей позиции.

### Сценарий 2. Рабочее место менеджера
При снятии позиции или отсутствии товара на складе менеджер получает список допустимых замен с объяснениями.

### Сценарий 3. Внутренний сервис каталогизации
Система помогает проставлять поле `Analogs` в номенклатуре, но не автоматически заменяет старые значения, а лишь рекомендует кандидатов с оценкой и причинами.

## 6.6 Производительность

Архитектура рассчитана на линейный скоринг по ограниченному числу кандидатов. Для текущего объёма подвыборки инференс выполняется в режиме реального времени.

Для каталога до 100 тысяч активных позиций рекомендуется:
- индексировать size_bucket;
- использовать sparse retrieval;
- использовать FAISS/hnswlib;
- кэшировать результаты по артикулу;
- ограничивать глубину перебора кандидатов;
- использовать асинхронный API и фоновую инициализацию каталога.

## 6.7 Качество кодовой базы

Модули типизированы, покрыты тестами, используют логирование и обработку ошибок. Пути реализованы через `pathlib`. Воспроизводимость обеспечивается фиксацией случайного состояния и снимком конфигурации.

---

# 7. Заключение, выводы и планы развития

## 7.1 Выполнение поставленных задач

Поставленные задачи выполнены на уровне прототипа и исследовательской версии.

Выполнено:

- разработана и обоснована архитектура подбора аналогов;
- построен конвейер предобработки промышленного датасета;
- реализована нормализация числовых характеристик с учётом единиц измерения;
- реализована нормализация категориальных и булевых признаков;
- согласован набор бизнес-характеристик;
- реализована бизнес-оценка с учётом важности и допустимых отклонений;
- реализован retrieval-слой с выдачей первых `K` кандидатов;
- сформированы метрики и артефакты;
- подготовлен ноутбук разведочного анализа;
- сформирована экспертная выборка;
- реализован FastAPI-сервис с объяснимым ответом;
- проведена повторная EDA-итерация;
- выполнены бизнес-метрики и метрики candidate generation;
- выполнено обоснование перехода к Плану D с негативными парами.

## 7.2 Ключевые выводы

### Вывод 1
Под подбором аналогов нельзя понимать простой текстовый поиск.  
Для климатического оборудования необходимо использовать нормализованные технические характеристики.

### Вывод 2
Поле `Analogs` не является достоверным эталоном в полном объёме,  
но в технической части показывает очень высокую согласованность с бизнес-логикой:

- `PassRateEvaluable = 0.9915`

Это означает, что поле можно использовать как слабый источник позитивов.

### Вывод 3
После переработки конвейера retrieval-качество выросло кратно:
- `MRR = 0.2692`
- `HitRate@10 = 0.6678`

### Вывод 4
Candidate generation работает хорошо на уровне запросов:
- `candidate_recall = 0.9854`

### Вывод 5
Бизнес-score реально оценивает кандидатов:
- `top1_pass_rate = 0.8933`

### Вывод 6
Система уже готова к роли объяснимого ассистента подбора аналогов.

### Главный вывод
Итоговая система является не просто текстовым поиском и не «чёрной коробкой», а гибридной объяснимой моделью, где:

- текст помогает искать кандидатов;
- параметры определяют допустимость;
- негативные пары проверяют качество разделения.

## 7.3 Ограничения работы

Использован единственный датасет без кросс-доменной проверки.  
Поведенческие данные не использовались.  
Глубокие текстовые модели не внедрены в рабочую версию.  
Качество рекомендаций ограничено полнотой характеристик.  
`hard_fail` пока слишком агрессивен для значительной части кандидатов.

## 7.4 Планы развития

### Направление данных
- расширять словарь характеристик;
- добавить парсинг технических PDF;
- улучшить unit-aware нормализацию мощности;
- обрабатывать явно Вт, кВт и неконсистентные обозначения;
- проверять мощность по размерному классу и площади;
- добавить источник данных для брендов и серий.

### Направление retrieval
- расширить candidate generation;
- добавить соседние размерные классы;
- использовать товары в диапазоне мощности ±15-20%;
- использовать товары в диапазоне площади ±15-20%;
- добавить индексирование по размерным классам;
- добавить vector index для текстовых эмбеддингов;
- отслеживать `CandidateRecall` как отдельный SLA retrieval-слоя.

### Направление алгоритмов
- добавить текстовую компоненту как отдельную часть финального скоринга;
- рассмотреть Sentence-Transformers для семантики названий;
- обучить метрическую модель на размеченных парах аналогов;
- добавить подбор весов бизнес-score через автоматическую оптимизацию;
- реализовать План D полностью:
  - генерация негативов;
  - AUROC / AUPRC;
  - калибровка порога;
- переработать `hard_fail`.

### Направление инфраструктуры
- мигрировать на векторный индекс;
- использовать FAISS или hnswlib;
- кэшировать результаты по артикулу;
- добавить фоновую переинициализацию каталога;
- подготовить Docker-образ.

### Направление продуктов
- блок похожие модели на карточке;
- поиск замены в личном кабинете;
- интеграция с CRM;
- автоподбор замены при отсутствии товара на складе;
- внутренний сервис для проставления аналогов.

### Направление оценки
- разметить экспертные пары запрос - аналог;
- построить Precision@K, Recall@K, NDCG@K на очищенной разметке;
- рассчитать бизнес-метрику продажи аналога;
- оценить долю успешных замен;
- оценить сокращение времени менеджера.

## 7.5 Личные результаты
В рамках дипломного проекта применены знания, полученные за два года обучения на курсе Data Scientist. Использованы модули по статистике, Python для анализа данных, SQL и базы данных, аналитике больших данных, математике, работе с признаками, нейронным сетям, рекомендательным системам, обработке естественного языка, временным рядам, компьютерному зрению и глубокому обучению.
Формирование производственного конвейера с артефактами, логированием, тестами и мониторингом стало естественным продолжением учебных модулей по инженерным практикам

---

# 8. Список литературы

## 8.1 Учебные и методические источники

Manning Christopher D., Raghavan Prabhakar, Schütze Hinrich. *Introduction to Information Retrieval*. Cambridge University Press, 2008.

Ricci Francesco, Rokach Lior, Shapira Bracha. *Recommender Systems Handbook*. Springer, 3rd edition, 2021.

Hastie Trevor, Tibshirani Robert, Friedman Jerome. *The Elements of Statistical Learning*. Springer, 2nd edition, 2009.

Géron Aurélien. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. O'Reilly, 3rd edition, 2022.

Kuhn Max, Johnson Kjersti. *Feature Engineering and Selection*. CRC Press, 2019.

## 8.2 Научные статьи и работы

Salton Gerard, Buckley Chris. Term-weighting approaches in automatic text retrieval. *Information Processing & Management*, 1988.

Robertson Stephen, Zaragoza Hugo. The Probabilistic Relevance Framework BM25 and Beyond. *Foundations and Trends in Information Retrieval*, 2009.

He Ruining, McAuley Julian. Ups and Downs: Modeling the Visual Evolution of Fashion Trends with One-Class Collaborative Filtering. *WWW*, 2016.

Lundberg Scott, Lee Su-In. A Unified Approach to Interpreting Model Predictions. *NeurIPS*, 2017.

Hancerliogullari Gokcen et al. Content-Based Recommendation Systems: A Review of Recent Developments. *ACM Computing Surveys*, 2022.

Pei Pei et al. Product Matching: A Survey. *Information Processing & Management*, 2023.

Johnson Jeff, Douze Matthijs, Jégou Hervé. Billion-scale similarity search with GPUs. *IEEE Transactions on Big Data*, 2019.

Malkov Yu. A., Yashunin D. A. Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs. *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 2020.

## 8.3 Документация инструментов

NumPy Documentation. numpy.org.

pandas Documentation. pandas.pydata.org.

scikit-learn Documentation. scikit-learn.org.

SciPy Documentation. scipy.org.

statsmodels Documentation. statsmodels.org.

FastAPI Documentation. fastapi.tiangolo.com.

Plotly Python Documentation. plotly.com/python.

ydata-profiling Documentation. ydata-profiling.readthedocs.io,.

Evidently Documentation. evidentlyml.com.

SHAP Documentation. shap.readthedocs.io.

Pydantic Documentation. docs.pydantic.dev.

FAISS Documentation. github.com/facebookresearch/faiss.

## 8.4 Курс и внутренние материалы

Нетология. *Программа курса Data Scientist расширенный курс, модули 1-16 и итоговый модуль*. Материалы курса, 2022–2026.

Нетология. *Методические рекомендации по подготовке и защите дипломного проекта*. 2026.

---

# 9. Приложения

## 9.1 Состав приложенного комплекта

Пояснительная записка в Jupyter ноутбук в формате Markdown.

EDA в Jupyter ноутбук.

Приложение : 
- Модули предобработки: `preprocessor`, `parameterizer`, `text_matcher`, `param_matcher`, `business_service`. 
- Модуль `config` с конфигурацией. 
- Модуль `data_loader`.
- Модуль `api` с реализацией FastAPI сервиса.
- Каталог `tests` с модульными и интеграционными тестами.
- Каталог `notebooks` с ноутбуками "Пояснительная записка", EDA.
- Каталог `artifacts` с моделями, матрицами, метриками, отчётами, визуализациями, метаданными запуска, снимком конфигурации, списком зависимостей с версиями.
- Документ `README` с описанием установки и использования.

---